In [38]:
# ==========================================================
# TASK 1 - DATA CLEANING & PREPARATION (SWYNEX Internship)
# Dataset: Netflix Movies and TV Shows
# ==========================================================

In [39]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [40]:
# ---------- 1. SETUP FILE PATHS ----------

DATA_PATH = Path("../data/Netflix_Raw_Data.csv")
OUTPUT_PATH = Path("../data/Netflix_Cleaned_Data.csv")

In [41]:
# ---------- 2. LOAD RAW DATA ----------

df = pd.read_csv(r"C:\Users\Eucloid\Desktop\SWYNEX-Data-Cleaning-Preparation\Raw Data\netflix_titles.csv")
print("Dataset Loaded Successfully")
print("Rows:", df.shape[0], "| Columns:", df.shape[1])

Dataset Loaded Successfully
Rows: 271 | Columns: 12


In [42]:
# Create a working copy so the original raw data stays unchanged

df_clean = df.copy()

In [43]:
# ---------- 3. IDENTIFY MISSING VALUES (overview) ----------

print("\n--- Missing values before cleaning ---")
print(df_clean.isnull().sum())


--- Missing values before cleaning ---
show_id          3
type             3
title            3
director        41
cast            39
country         17
date_added      21
release_year    11
rating          14
duration        16
listed_in        3
description      8
dtype: int64


In [44]:
# ---------- 4. DROP FULLY EMPTY ROWS ----------

df_clean = df_clean.dropna(how="all")
print("\nShape after dropping empty rows:", df_clean.shape)


Shape after dropping empty rows: (268, 12)


In [45]:
# ---------- 5. FILL MISSING VALUES (column-wise) ----------

df_clean["listed_in"] = df_clean["listed_in"].fillna("Unknown")
df_clean["description"] = df_clean["description"].fillna("No description available")
df_clean["director"] = df_clean["director"].fillna("Unknown")
df_clean["cast"] = df_clean["cast"].fillna("Unknown")
df_clean["country"] = df_clean["country"].fillna(df_clean["country"].mode()[0])
df_clean["rating"] = df_clean["rating"].fillna(df_clean["rating"].mode()[0])

In [46]:
# ---------- 6. FIX release_year (numeric type) ----------
df_clean["release_year"] = pd.to_numeric(df_clean["release_year"], errors="coerce")
df_clean["release_year"] = df_clean["release_year"].fillna(df_clean["release_year"].median())


In [47]:
# ---------- 7. FIX date_added (datetime type) ----------
df_clean["date_added"] = pd.to_datetime(df_clean["date_added"], errors="coerce")
df_clean["date_added"] = df_clean["date_added"].fillna(df_clean["date_added"].mode()[0])

In [48]:
# ---------- 8. CLEAN 'type' COLUMN (casing/spaces/invalid text fix) ----------
df_clean["type"] = df_clean["type"].astype(str).str.strip().str.lower()
df_clean["type"] = df_clean["type"].replace({"none": np.nan, "nan": np.nan})
df_clean["type"] = df_clean["type"].map({"movie": "Movie", "tv show": "TV Show"})
df_clean["type"] = df_clean["type"].fillna("Movie")   # agar koi type missing reh gaya

print("\n--- Type column after cleaning ---")
print(df_clean["type"].value_counts(dropna=False))


--- Type column after cleaning ---
type
Movie      182
TV Show     86
Name: count, dtype: int64


In [49]:
# ---------- 9. SPLIT 'duration' INTO duration_minutes / duration_seasons ----------

df_clean["duration_num"] = df_clean["duration"].astype(str).str.extract(r"(\d+)").astype(float)

df_clean["duration_minutes"] = df_clean.apply(
    lambda row: row["duration_num"] if row["type"] == "Movie" else None, axis=1
)
df_clean["duration_seasons"] = df_clean.apply(
    lambda row: row["duration_num"] if row["type"] == "TV Show" else None, axis=1
)


In [50]:
# ---------- 10. FILL MISSING duration_minutes / duration_seasons ----------

movie_median = df_clean.loc[df_clean["type"] == "Movie", "duration_minutes"].median()
df_clean.loc[df_clean["type"] == "Movie", "duration_minutes"] = (
    df_clean.loc[df_clean["type"] == "Movie", "duration_minutes"].fillna(movie_median)
)

tv_median = df_clean.loc[df_clean["type"] == "TV Show", "duration_seasons"].median()
df_clean.loc[df_clean["type"] == "TV Show", "duration_seasons"] = (
    df_clean.loc[df_clean["type"] == "TV Show", "duration_seasons"].fillna(tv_median)
)

df_clean = df_clean.drop(columns=["duration", "duration_num"], errors="ignore")

print("\n--- Missing values after all fills ---")
print(df_clean.isnull().sum())


--- Missing values after all fills ---
show_id               0
type                  0
title                 0
director              0
cast                  0
country               0
date_added            0
release_year          0
rating                0
listed_in             0
description           0
duration_minutes     86
duration_seasons    182
dtype: int64


In [51]:
# ---------- 11. REMOVE EXACT DUPLICATE ROWS ----------

df_clean = df_clean.drop_duplicates()
print("\nShape after removing duplicates:", df_clean.shape)



Shape after removing duplicates: (250, 13)


In [52]:
# ---------- 12. FIX INVALID / INCONSISTENT VALUES ----------

# 12a. Future release years (> 2026) -> replace with median
valid_median_year = df_clean.loc[df_clean["release_year"] <= 2026, "release_year"].median()
df_clean.loc[df_clean["release_year"] > 2026, "release_year"] = valid_median_year

# 12b. 0 or negative duration for movies -> replace with median movie duration

valid_movie_median = df_clean.loc[
    (df_clean["type"] == "Movie") & (df_clean["duration_minutes"] > 0), "duration_minutes"
].median()
df_clean.loc[
    (df_clean["type"] == "Movie") & (df_clean["duration_minutes"] <= 0), "duration_minutes"
] = valid_movie_median

# 12c. Verify fixes

print("\nRows with future release_year now:", (df_clean["release_year"] > 2026).sum())
print("Movies with 0/negative duration now:",
      ((df_clean["type"] == "Movie") & (df_clean["duration_minutes"] <= 0)).sum())


Rows with future release_year now: 0
Movies with 0/negative duration now: 0


In [53]:
# ---------- 13. FINAL CHECK ----------

print("\n--- FINAL DATASET SUMMARY ---")
print("Final shape:", df_clean.shape)
print("Total missing values:", df_clean.isnull().sum().sum())


--- FINAL DATASET SUMMARY ---
Final shape: (250, 13)
Total missing values: 250


In [54]:
import os

output_path = r"C:\Users\Eucloid\Desktop\SWYNEX-Data-Cleaning-Preparation\output\Netflix_Cleaned_Data.csv"

output_dir = os.path.dirname(r"C:\Users\Eucloid\Desktop\SWYNEX-Data-Cleaning-Preparation\output\Netflix_Cleaned_Data.csv")
os.makedirs(output_dir, exist_ok=True)

df_clean.to_csv(output_path, index=False)
print("\nFinal cleaned data saved successfully at:", os.path.abspath(r"C:\Users\Eucloid\Desktop\SWYNEX-Data-Cleaning-Preparation\output\Netflix_Cleaned_Data.csv"))


Final cleaned data saved successfully at: C:\Users\Eucloid\Desktop\SWYNEX-Data-Cleaning-Preparation\output\Netflix_Cleaned_Data.csv
